# **ThermoModel v2: Bands That Account for Diminishing Returns**

The goal is unchanged: a corridor whose upper edge marks cycle tops and whose lower edge marks cycle bottoms. What changes is how the bands are built, because the original model has two structural problems that show up exactly where it hurts - predicting tops.

## What was wrong

The original ThermoModel fits a **cubic polynomial in day index** to log price, then adds **fixed offsets in log space** for the bands.

1. A cubic in raw time extrapolates explosively upward. That is the *opposite* of diminishing returns, and it is why the upper band runs away: run out-of-sample on the 2020-2025 cycle, the original approach projected a top near **$495,000** against an actual **$125,000**.
2. A fixed offset in log space means the band keeps a constant multiplicative width forever. If cycle amplitude is compressing, no fixed offset can track it - too wide late, or too narrow early. In 2017 the same approach put the upper band *below* the actual top (-39%), so price simply broke through it.

## The trend: a power law whose exponent eases off

The backbone is a log-log trend, with `L = log10(days since genesis)`:

    log10(price) = a + b·L + c·L²

A plain power law (`c = 0`) has one fixed exponent. Refitted in real time, that exponent has kept falling - 6.8 at the 2015 bottom, 5.9 at 2018, 5.8 at 2022, 5.6 today - so every fit projected the next cycle too high: forecast from the previous bottom, it overshot the 2018, 2022 and 2026 bottoms by +130%, +44% and +47%. The quadratic term builds that decline in: the exponent eases from about 5.8 in 2014 to 5.3 in 2018 and 4.8 today (4.3 by 2045; it never reaches zero on any horizon that matters). The trend grows ~30% a year today and ~24% by 2030, against 36% and 30% for the fixed exponent.

On the two cycles with enough history behind the fit, the concave trend forecast the next bottom four years ahead at -7% and +22%, and - through v2's upper band - the next top at -1% and +13%, where the fixed exponent overshot by +22% and +44% (see `backtest_trend()` and the validation below). It does need that history: fitted before 2018, on under eight years of data, it is unstable. In-sample the two fits are all but identical (R² 0.964 vs 0.962). `load_dataframe(concave_trend=False)` restores the plain power law.

## What the data actually says

Measured against that trend, the cycle extremes are:

| Cycle top | multiple of trend | | Cycle bottom | multiple of trend |
|---|---|---|---|---|
| 2011-06 | 18.5x | | 2011-11 | 0.46x |
| 2013-12 | 9.6x | | 2015-01 | 0.46x |
| 2017-12 | 5.7x | | 2018-12 | 0.54x |
| 2021-11 | 3.0x | | 2022-11 | 0.48x |
| 2025-10 | 1.5x | | 2026-11 | 0.52x |

**Tops compress hard.** The bottom multiples above look flat, but they are measured on the day of the bottom *in price*, which is not the day price sat lowest *against the trend*. Measured that way - the deepest close of each cycle - the floor tightens too: **0.32x, 0.32x, 0.42x, 0.48x, 0.51x**.

That distinction was a real bug. A lower band set at the mean of the five bottom multiples (≈ 0.49x) is a central estimate of where a bottom lands, not a floor, and it did not hold: **864 closes, 14% of the history, fell below it, the deepest 34% below**. Most of them were nowhere near a bottom - they are the long flat stretches of 2012 and 2015-16, where price idled while the trend climbed past it.

## The v2 construction

    upper(t) = trend(t) + max(a + b·cycle_position(t), upper_floor)
    lower(t) = trend(t) + min(c + d·cycle_position(t), lower_cap)

Both offsets are linear in cycle position. The upper one decays, fitted on the five observed tops. The lower one rises, fitted by **quantile regression of the daily closes against the trend** at `LOWER_TAU = 0.005` - a support rather than an average: **99.5% of the history closes above it and the deepest breach is 6%** instead of 34%, with the last two cycle bottoms landing on the band itself (1.6% above it and 4.2% below it, in corridor width).

The floor stops the decaying upper band from extrapolating *below* the trend, which would be nonsense. At its default of 0 it binds on no historical cycle, so it costs nothing in backtest. The cap is its mirror: `"last"`, the default, freezes the lower band where it stands on the last day of actual data, so the tightening is read off history but never extrapolated - a rising floor under a decaying ceiling would otherwise squeeze the corridor shut.

## Validation

Leave-future-out, on the task the model exists for: for each cycle top, refit the trend and the decay on data and cycles strictly *before* it, then compare the predicted upper band against the price actually reached. The original fixed-offset approach, and v2 on a fixed-exponent power law, go through the identical test as baselines.

**Result: median absolute error 13% (v2) vs 22% (v2 on a fixed-exponent power law) vs 83% (original).** On the 2021 and 2025 tops v2 lands at -1% and +13%, where the fixed exponent overshot by +22% and +44% and the original by +83% and ~4x. The one large miss is 2017 (-79%): refitted on data up to mid-2017, only seven years of it, the concave trend bends too early - the same short-history instability the bottom backtest shows.

## The honest limitation

The fitted decay is steep enough that the upper band settles onto its asymptote - close to the trend - from around early 2028. The decay bends into that asymptote smoothly (a softplus rather than a hard clamp, which would put a visible kink in the band on the day the two met and a dead flat line after it). That is the faithful reading of five data points, but five is not many, and "future tops sit near the trend line" is an aggressive claim. `upper_floor` is exposed so the band can be given a positive asymptote instead (0.1 → tops at least 1.26x trend, 0.2 → 1.58x). Raising it changes no historical band, only the extrapolation.

*Not financial advice.*


In [1]:
import sys
import sys, os
_base = os.getcwd()
while _base and not os.path.isdir(os.path.join(_base, 'functions')):
    _parent = os.path.dirname(_base)
    if _parent == _base: break
    _base = _parent
if os.path.abspath(_base) not in sys.path:
    sys.path.insert(0, os.path.abspath(_base))
from functions.functions_ThermoModel2 import (
    load_dataframe,
    plot_thermomodel,
    validate_top_prediction,
    backtest_trend,
    validate_lower_band,
    project_next_top,
)
import warnings
warnings.filterwarnings("ignore")

df, params = load_dataframe()

✅ BTC CoinMetrics: 6536 rows (last: 2026-05-24)


✅ BTC CoinMetrics Community API: 188 genuine daily rows; transactions through 2026-11-25, active addresses through 2026-11-25


## The bands

In [2]:
plot_thermomodel(df, params)

Trend: log10(price) = -26.283 +11.589*L -0.8840*L^2,  L = log10(days since genesis)   R2 = 0.964
Local exponent today: 4.84
Top multiples vs trend, cycle by cycle: [18.47, 9.65, 5.69, 2.99, 1.48] (compressing)
Bottom multiples vs trend:              [0.46, 0.46, 0.54, 0.48, 0.52]
Deepest close of each cycle vs trend:   [0.32, 0.32, 0.42, 0.48, 0.51] (the floor the lower band holds; tightening)

Today (2026-11-25): price $60,268  band $62,719 - $160,513
Position in band: -4.6% smoothed (-4.2% raw) - the chart's readout box and the oscillator panel both show the smoothed value, as the original does.
In one year (2027-11-25): band $82,845 - $188,346   corridor midpoint $124,914


## Does the lower band hold?

The old flat band and the fitted support, measured against every close in the history: how many fell through, how far the deepest one went, and where each cycle bottom ends up in the corridor once the band is in place.

In [3]:
lower_band = validate_lower_band(df, params)

Does the lower band hold?

                     rule  days below  % of history deepest breach   band today
v2 flat (mean of bottoms)         864          14.5         0.661x 0.490x trend
   v2 support (tau=0.005)          29           0.5         0.942x 0.539x trend

Deepest close of each cycle, against the trend - what the band has to clear
  [0.32, 0.32, 0.42, 0.48, 0.51]   (the bottoms themselves: [0.46, 0.46, 0.54, 0.48, 0.52])

Where each cycle bottom sits in the corridor (0 = on the lower band)
  2011-11-18    12.2%
  2015-01-14    10.5%
  2018-12-15    13.0%
  2022-11-21     1.6%
  2026-11-21    -4.2%


## Validation: out-of-sample top prediction, v2 vs the original

In [4]:
validation = validate_top_prediction(df)

✅ BTC CoinMetrics: 6536 rows (last: 2026-05-24)


✅ BTC CoinMetrics Community API: 188 genuine daily rows; transactions through 2026-11-25, active addresses through 2026-11-25


Leave-future-out cycle top prediction

 cycle top  actual  v2 predicted  v2 error %  v2 power law predicted  v2 power law error %  v1 predicted  v1 error %
2017-12-17  19,250         3,981         -79                  20,687                     8        11,726         -39
2021-11-09  67,096        66,546          -1                  81,975                    22       122,557          83
2025-10-06 124,824       141,449          13                 179,498                    44       494,811         296

Median |error|:  v2 (concave trend) 13.3%   v2 (power-law trend) 22.2%   v1 (fixed offset) 82.7%


## Why the trend is concave: every bottom, forecast in real time

Each cycle bottom forecast with only the data available at the time, from the previous bottom (about four years ahead, where the model stands today) and from the top before it (about one year ahead). The bottom is forecast as the trend times the mean bottom multiple so far - the corridor's central estimate of a bottom. Since the lower band became a support it sits below that line; what is tested here is the trend underneath both.

In [5]:
trend_backtest = backtest_trend()

✅ BTC CoinMetrics: 6536 rows (last: 2026-05-24)


✅ BTC CoinMetrics Community API: 188 genuine daily rows; transactions through 2026-11-25, active addresses through 2026-11-25


Real-time forecasts of each cycle bottom from the mean bottom multiple

    bottom                                   forecast actual  power law error % power law exponent  concave error % concave exponent
2018-12-15 ~4y, from the previous bottom (2015-01-14)  3,185              130.4               6.82            285.6             7.43
2018-12-15      ~1y, from the top before (2017-12-17)  3,185              -12.9               5.70            -57.2             3.98
2022-11-21 ~4y, from the previous bottom (2018-12-15) 15,778               44.4               5.88             -6.8             5.17
2022-11-21      ~1y, from the top before (2021-11-09) 15,778               43.7               5.87             31.0             5.63
2026-11-21 ~4y, from the previous bottom (2022-11-21) 60,123               47.3               5.84             22.3             5.49
2026-11-21      ~1y, from the top before (2025-10-06) 60,123               30.2               5.70              3.5             5.

## Where does this put the next cycle top?

In [6]:
project_next_top(df, params)

Decay of the top multiple: offset = 1.405 -0.293 * cycle_position  (floored at 0.00)

Implied cycle top if it lands on:
  Oct 2028:  trend $   186,734   upper band $    217,317   (offset +0.066 = 1.16x trend) (near the asymptote)
  Oct 2029:  trend $   236,086   upper band $    259,323   (offset +0.041 = 1.10x trend) (near the asymptote)
  Oct 2030:  trend $   294,691   upper band $    311,633   (offset +0.024 = 1.06x trend) (near the asymptote)

Read this with care. By the next cycle the linear decay has run below its asymptote,
so the band is settling onto `upper_floor` rather than still tracking the decay -
these figures are driven mostly by the trend extrapolation, and the decay itself
rests on five observations. Raise `upper_floor` in load_dataframe() for a less
aggressive assumption about how far tops keep compressing.


## A less aggressive assumption

The same model with a positive asymptote on the upper band, so that future tops are not assumed to converge all the way onto the trend line. Historical bands are identical; only the extrapolation moves.

In [7]:
df_soft, params_soft = load_dataframe(upper_floor=0.15)
project_next_top(df_soft, params_soft)

✅ BTC CoinMetrics: 6536 rows (last: 2026-05-24)


✅ BTC CoinMetrics Community API: 188 genuine daily rows; transactions through 2026-11-25, active addresses through 2026-11-25


Decay of the top multiple: offset = 1.405 -0.293 * cycle_position  (floored at 0.15)

Implied cycle top if it lands on:
  Oct 2028:  trend $   186,734   upper band $    278,576   (offset +0.174 = 1.49x trend) (near the asymptote)
  Oct 2029:  trend $   236,086   upper band $    344,196   (offset +0.164 = 1.46x trend) (near the asymptote)
  Oct 2030:  trend $   294,691   upper band $    423,826   (offset +0.158 = 1.44x trend) (near the asymptote)

Read this with care. By the next cycle the linear decay has run below its asymptote,
so the band is settling onto `upper_floor` rather than still tracking the decay -
these figures are driven mostly by the trend extrapolation, and the decay itself
rests on five observations. Raise `upper_floor` in load_dataframe() for a less
aggressive assumption about how far tops keep compressing.


## Timing

The bands say *how high and how low*, not *when*. The ThermoClock, which combines them with the cycle timing from `2a.Cycles` into a dated price path, lives in its own notebook: `1j.ThermoClock`.